In [1]:

import pykrige
import sklearn as sl
import statsmodels as sm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import tensorly as tl
import joblib
import xgboost

In [2]:
"""
Correlazioni stimate separatamente
  - temporale: coppie della STESSA stazione a sfasamenti di 1,...,K giorni
    (autocorrelazione empirica di una serie storica)
  - spaziale: coppie di stazioni DIVERSE nello STESSO giorno (o
    entro una piccola finestra temporale), sulle distanze reali in km
"""

from dataclasses import dataclass
from typing import Optional

import numpy as np
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from joblib import Parallel, delayed


EARTH_RADIUS_KM = 6371.0088

# Componente larga scala (comune per RF, SVM, XGBoost)

def build_large_scale_model(model_type: str, model_params: dict, n_predictors: int):
    """
    Costruisce il modello per la componente di larga scala S(s,t), con
    interfaccia comune (.fit(X,y), .predict(X)) indipendentemente dal
    tipo scelto (la stima del variogramma e il kriging dei residui sono indipendenti da questa scelta)

    model_type : {"rf", "svr", "xgboost"}
    "rf": RandomForestRegressor. Robusta, poca calibrazione, interazioni non lineari e non serve standardizzare i predittori
    "svr": Support Vector Regression con nucleo radiale RBF, dentro una procedura con StandardScaler. Se i dati di addestramento sono molto più numerosi, sottocampiona le righe di addestramento
    "xgboost": XGBRegressor. Migliore della RF su dati tabellari, addestramento più rapido grazie al boosting. Più sensibile del RF a overfitting se n_estimators/max_depth non sono regolati
    """
    model_type = model_type.lower()

    if model_type == "rf":
        default_params = dict(n_estimators=300, max_features=n_predictors // 3,min_samples_leaf=10, oob_score=True, random_state=13, n_jobs=-1,)
        default_params.update(model_params)
        return RandomForestRegressor(**default_params)

    if model_type == "svr":
        default_svr_params = dict(kernel="rbf", C=10.0, epsilon=0.5, gamma="scale",max_train_rows=15000,)
        default_svr_params.update(model_params)
        max_train_rows = default_svr_params.pop("max_train_rows")

        svr = SVR(**default_svr_params)
        pipe = Pipeline([("scaler", StandardScaler()), ("svr", svr)])
        pipe._svr_max_train_rows = max_train_rows
        return pipe

    if model_type == "xgboost":
        default_xgb_params = dict(n_estimators=300, max_depth=8, learning_rate=0.01,subsample=0.6, colsample_bytree=0.8,random_state=13, n_jobs=-1,)
        default_xgb_params.update(model_params)
        return XGBRegressor(**default_xgb_params)

    raise ValueError(f"model_type sconosciuto: {model_type!r}. Usa 'rf', 'svr' o 'xgboost'.")


def _get_oob_score(model) -> Optional[float]:
    """Restituisce il punteggio OOB se disponibile (solo RandomForestRegressor con oob_score=True)"""
    inner = model
    if isinstance(model, Pipeline):
        inner = model.named_steps.get("svr", model)
    return getattr(inner, "oob_score_", None)


# Proiezione metrica locale (equirettangolare) lat/lon -> km

def latlon_to_km(lat: np.ndarray, lon: np.ndarray, lat0: Optional[float] = None, lon0: Optional[float] = None) -> tuple[np.ndarray, np.ndarray, float, float]:
    """
    Converte lat/lon in coordinate cartesiane locali in km, con proiezione equirettangolare centrata sul baricentro dei punti
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    if lat0 is None:
        lat0 = float(lat.mean())
    if lon0 is None:
        lon0 = float(lon.mean())
    lat0_rad = np.radians(lat0)
    x_km = EARTH_RADIUS_KM * np.radians(lon - lon0) * np.cos(lat0_rad)
    y_km = EARTH_RADIUS_KM * np.radians(lat - lat0)
    return x_km, y_km, lat0, lon0

# Registro stazioni (coordinate metriche + matrice di distanza precalcolata)
class StationRegistry:

    def __init__(self, station_ids_unique: np.ndarray, station_coords_km: np.ndarray):
        station_ids_unique = np.asarray(station_ids_unique)
        station_coords_km = np.asarray(station_coords_km, dtype=float)
        if len(station_ids_unique) != len(station_coords_km):
            raise ValueError("station_ids_unique e station_coords_km devono avere la stessa lunghezza")

        self.station_ids = station_ids_unique
        self.coords = station_coords_km
        self.n_stations = len(station_ids_unique)
        self._id_to_idx = {sid: i for i, sid in enumerate(station_ids_unique)}
        self.dist_matrix = cdist(station_coords_km, station_coords_km)  # km
        self._tree = cKDTree(station_coords_km)

    def ids_to_indices(self, station_id_per_row: np.ndarray) -> np.ndarray:
        return np.array([self._id_to_idx[sid] for sid in station_id_per_row])

    def spatial_distance(self, idx_a: np.ndarray, idx_b: np.ndarray) -> np.ndarray:
        return self.dist_matrix[idx_a, idx_b]

    def query_knn(self, coord: np.ndarray, k: int) -> np.ndarray:
        k = min(k, self.n_stations)
        _, idx = self._tree.query(coord, k=k)
        return np.atleast_1d(idx)


# Covarianza separabile: rho(h,u) = rho_s(h) * rho_t(u)

@dataclass
class SeparableExponentialCovariance:
    """
    C(h, u) = partial_sill * exp(-h/theta_s) * exp(-u/theta_t)
    con nugget aggiunto solo sulla diagonale (h=0, u=0) in fase di kriging.
    """
    total_var: float = 1.0
    nugget: float = 0.0
    theta_s: float = 100.0
    theta_t: float = 10.0

    @property
    def partial_sill(self) -> float:
        return max(self.total_var - self.nugget, 1e-9)

    def covariance(self, h: np.ndarray, u: np.ndarray) -> np.ndarray:
        return self.partial_sill * np.exp(-h / self.theta_s) * np.exp(-u / self.theta_t)

    @property
    def practical_theta_s(self) -> float:
        return 3.0 * self.theta_s

    @property
    def practical_theta_t(self) -> float:
        return 3.0 * self.theta_t


def _fit_exponential_decay(lags: np.ndarray, corr: np.ndarray, weights: np.ndarray,max_range_abs: Optional[float] = None) -> float:
    """
    Stima theta in una correlazione esponenziale corr(lag) = exp(-lag/theta)
    tramite minimi quadrati pesati sul numero di coppie per sfasamento.
    Usa solo lag con correlazione positiva
    """
    lags = np.asarray(lags, dtype=float)
    corr = np.asarray(corr, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = (lags > 0) & np.isfinite(corr) & (weights > 0)
    if valid.sum() < 2:
        return max_range_abs if max_range_abs is not None else float(np.max(lags[lags > 0], initial=1.0))

    lags_v, corr_v, w_v = lags[valid], corr[valid], weights[valid]

    def model(lag, rng):
        return np.exp(-lag / rng)

    p0 = [max(lags_v[0], 1e-3)]
    bounds = (1e-3, max_range_abs if max_range_abs is not None else np.inf)

    try:
        popt, _ = curve_fit(model, lags_v, corr_v, p0=p0, bounds=bounds, sigma=1.0 / np.sqrt(w_v),)
        return float(popt[0])
    except Exception:
        # alternativa robusta: stima theta dal primo lag con corr > 1/e,
        # oppure dal decadimento medio (metodo dei momenti)
        above = lags_v[corr_v > np.exp(-1)]
        if len(above) > 0:
            return float(above.max())
        return float(np.clip(lags_v[0], 1e-3, max_range_abs or lags_v[0]))


def fit_marginal_temporal(residuals: np.ndarray, station_idx: np.ndarray, time: np.ndarray,max_lag: int = 30, min_pairs_per_lag: int = 20, max_theta_t_abs: Optional[float] = 60.0,) -> tuple[float, np.ndarray, np.ndarray, np.ndarray]:
    """
    Stima la correlazione temporale marginale usando coppie della
    STESSA stazione a sfasamenti interi 1,...,max_lag giorni, poi adatta un
    decadimento esponenziale.

    restituisce theta_t, lags, corr_empirica, n_coppie_per_lag
    """
    residuals = np.asarray(residuals, dtype=float)
    station_idx = np.asarray(station_idx, dtype=int)
    time = np.asarray(time, dtype=float)

    lags_list = np.arange(1, max_lag + 1)
    corr_emp = np.full(len(lags_list), np.nan)
    n_pairs = np.zeros(len(lags_list), dtype=int)

    total_var = np.var(residuals)

    for st in np.unique(station_idx):
        rows = np.where(station_idx == st)[0]
        if len(rows) < 3:
            continue
        order = np.argsort(time[rows])
        t_sorted = time[rows][order]
        r_sorted = residuals[rows][order]

        # mappa tempo intero -> valore, per gestire buchi nella serie
        t_int = np.round(t_sorted).astype(int)
        val_map = dict(zip(t_int, r_sorted))

        for li, lag in enumerate(lags_list):
            keys = list(val_map.keys())
            for t0 in keys:
                t1 = t0 + lag
                if t1 in val_map:
                    prod = val_map[t0] * val_map[t1]
                    if np.isnan(corr_emp[li]):
                        corr_emp[li] = 0.0
                    corr_emp[li] += prod
                    n_pairs[li] += 1

    with np.errstate(invalid="ignore", divide="ignore"):
        corr_emp = np.where(n_pairs > 0, corr_emp / np.maximum(n_pairs, 1) / total_var, np.nan)

    keep = n_pairs >= min_pairs_per_lag
    theta_t = _fit_exponential_decay(lags_list[keep], corr_emp[keep], n_pairs[keep], max_range_abs=max_theta_t_abs)
    return theta_t, lags_list, corr_emp, n_pairs


def fit_marginal_spatial(residuals: np.ndarray, station_idx: np.ndarray, time: np.ndarray,registry: StationRegistry, time_tolerance: float = 5., n_bins_s: int = 20,min_pairs_per_bin: int = 20, max_theta_s_abs: Optional[float] = None,) -> tuple[float, np.ndarray, np.ndarray, np.ndarray]:
    """
    Stima la correlazione spaziale marginale usando coppie di stazioni
    diverse osservate nello stesso istante (o entro `time_tolerance`
    giorni), sulle distanze reali in km, poi adatta un decadimento
    esponenziale.

    restituisce theta_s, bin_centers_km, corr_empirica, n_coppie_per_bin
    """
    residuals = np.asarray(residuals, dtype=float)
    station_idx = np.asarray(station_idx, dtype=int)
    time = np.asarray(time, dtype=float)
    total_var = np.var(residuals)

    # raggruppa le righe per giorno intero (arrotondato)
    t_int = np.round(time).astype(int)
    unique_days = np.unique(t_int)

    prod_sum_h = []
    dist_h = []

    rng = np.random.default_rng(13)
    # per efficienza, se ci sono molti giorni, sottocampiona i giorni
    if len(unique_days) > 1200: #modificato 500 in 1200
        unique_days = rng.choice(unique_days, size=1200, replace=False)

    for day in unique_days:
        rows = np.where(t_int == day)[0]
        if len(rows) < 2:
            continue
        st_rows = station_idx[rows]
        r_rows = residuals[rows]
        # tutte le coppie di stazioni distinte in questo giorno
        Hs = registry.spatial_distance(st_rows[:, None], st_rows[None, :])
        Prod = np.outer(r_rows, r_rows)
        iu = np.triu_indices(len(rows), k=1)
        dist_h.append(Hs[iu])
        prod_sum_h.append(Prod[iu])

    if len(dist_h) == 0:
        raise ValueError("Nessuna coppia spaziale trovata: dati insufficienti nello stesso giorno.")

    dist_h = np.concatenate(dist_h)
    prod_h = np.concatenate(prod_sum_h)

    max_dist = np.percentile(dist_h, 75)
    # Bin log-spaziati invece che lineari
    positive = dist_h[dist_h > 0]
    d_min = max(np.percentile(positive, 1), max_dist * 1e-4) if len(positive) > 0 else max_dist * 1e-4
    d_min = min(d_min, max_dist * 0.3)
    bins = np.concatenate([[0.0], np.logspace(np.log10(d_min), np.log10(max_dist), n_bins_s)])
    bin_idx = np.clip(np.digitize(dist_h, bins) - 1, 0, n_bins_s - 1)

    corr_emp = np.full(n_bins_s, np.nan)
    n_pairs = np.zeros(n_bins_s, dtype=int)
    for b in range(n_bins_s):
        mask = bin_idx == b
        if mask.sum() > 0:
            corr_emp[b] = prod_h[mask].mean() / total_var
            n_pairs[b] = mask.sum()

    centers = (bins[:-1] + bins[1:]) / 2
    keep = n_pairs >= min_pairs_per_bin
    theta_s = _fit_exponential_decay(centers[keep], corr_emp[keep], n_pairs[keep], max_range_abs=max_theta_s_abs)
    return theta_s, centers, corr_emp, n_pairs


# Kriging locale spazio-temporale vincolato
class LocalSpatioTemporalKriging:
    """
    Kriging semplice (media 0 sui residui) spazio-temporale con
    covarianza separabile e intorni locali
    """

    def __init__(self, cov: SeparableExponentialCovariance, max_neighbors: int = 200,time_window: Optional[float] = 30.0, n_jobs: int = -1):
        self.cov = cov
        self.max_neighbors = max_neighbors
        self.time_window = time_window
        self.n_jobs = n_jobs

        self._residuals = None
        self._station_idx = None
        self._time = None
        self._registry = None
        self._rows_by_station = {}

    def fit(self, residuals, station_idx, time, registry):
        self._residuals = np.asarray(residuals, dtype=float)
        self._station_idx = np.asarray(station_idx, dtype=int)
        self._time = np.asarray(time, dtype=float)
        self._registry = registry
        self._rows_by_station = {st: np.where(self._station_idx == st)[0] for st in np.unique(self._station_idx)}
        return self

    def _select_neighbors(self, target_station_idx: int, target_time: float) -> np.ndarray:
        registry = self._registry
        target_coord = registry.coords[target_station_idx]
        nearest_stations = registry.query_knn(target_coord, k=registry.n_stations)

        rows_collected = []
        total = 0
        for st in nearest_stations:
            rows_st = self._rows_by_station.get(st)
            if rows_st is None or len(rows_st) == 0:
                continue
            if self.time_window is not None:
                mask_t = np.abs(self._time[rows_st] - target_time) <= self.time_window
                rows_st = rows_st[mask_t]
                if len(rows_st) == 0:
                    continue
            rows_collected.append(rows_st)
            total += len(rows_st)
            if total >= self.max_neighbors:
                break

        if not rows_collected:
            return np.array([], dtype=int)

        rows = np.concatenate(rows_collected)
        if len(rows) > self.max_neighbors:
            order = np.argsort(np.abs(self._time[rows] - target_time))
            rows = rows[order[: self.max_neighbors]]
        return rows

    def _predict_one(self, st_k: int, t_k: float) -> tuple[float, float]:
        registry = self._registry
        cov = self.cov

        rows = self._select_neighbors(st_k, t_k)
        if len(rows) == 0:
            return 0.0, cov.total_var

        obs_station_idx = self._station_idx[rows]
        obs_time = self._time[rows]
        obs_resid = self._residuals[rows]
        n_obs = len(rows)

        Hs_obs = registry.spatial_distance(obs_station_idx[:, None], obs_station_idx[None, :])
        Ht_obs = np.abs(obs_time[:, None] - obs_time[None, :])
        C = cov.covariance(Hs_obs, Ht_obs)
        np.fill_diagonal(C, cov.total_var)  # partial_sill + nugget sulla diagonale

        h0 = registry.spatial_distance(obs_station_idx, np.full(n_obs, st_k))
        u0 = np.abs(obs_time - t_k)
        c0 = cov.covariance(h0, u0)

        # ridge minimo per stabilita' numerica
        C_reg = C + np.eye(n_obs) * 1e-8 * cov.total_var

        try:
            weights = np.linalg.solve(C_reg, c0)
        except np.linalg.LinAlgError:
            weights = np.linalg.lstsq(C_reg, c0, rcond=None)[0]

        pred = float(np.dot(weights, obs_resid))
        var = float(cov.total_var - np.dot(weights, c0))
        return pred, max(var, 0.0)

    def predict(self, target_station_idx: np.ndarray, target_time: np.ndarray):
        target_station_idx = np.asarray(target_station_idx, dtype=int)
        target_time = np.asarray(target_time, dtype=float)
        m = len(target_station_idx)

        if  m > 50:
            results = Parallel(n_jobs=self.n_jobs, prefer="threads")(delayed(self._predict_one)(target_station_idx[k], target_time[k]) for k in range(m))
            preds, variances = zip(*results)
            return np.array(preds), np.array(variances)

        preds = np.empty(m)
        variances = np.empty(m)
        for k in range(m):
            preds[k], variances[k] = self._predict_one(target_station_idx[k], target_time[k])
        return preds, variances


# Modello RFSTK completo
class RFSTK:
    """
    Random Forest Spatio-Temporal Kriging con:
      1. modello scelto per la componente large-scale S(s,t).
      2. Variogramma separabile stimato da correlazioni marginali
         temporale: stessa stazione, lag 1,...,K giorni, 
         spaziale: stazioni diverse, stesso giorno
      3. Kriging semplice locale sui residui.
      4. noltiplicatore lambda in [0,1] sulla correzione di kriging, scelto
         per cross-validation (LOSOCV) in modo da non
         peggiorare l'R2 rispetto alla sola random forest.
    """

    def __init__(self, model_type: str = "rf", model_params: Optional[dict] = None,
                 rf_params: Optional[dict] = None,
                 temporal_params: Optional[dict] = None,
                 spatial_params: Optional[dict] = None,
                 kriging_params: Optional[dict] = None,
                 lambda_grid: Optional[np.ndarray] = None,
                 cv_frac_stations: float = 0.2,
                 cv_seed: int = 123,
                 use_km_projection: bool = True,
                 fixed_lambda: Optional[float] = None):
        """
        fixed_lambda: Se specificato (in [0, 1]), disattiva la selezione di lambda per cross-validation e usa sempre questo valore fisso 
        (fixed_lambda=1.0 per applicare il kriging al 100% senza alcun filtro di sicurezza, fixed_lambda=0.0 per disattivarlo del tutto)
        model_type: {"rf", "svr", "xgboost"}. Modello usato per la componente di larga scala S(s,t)
        model_params: Parametri passati al costruttore del modello scelto (RandomForestRegressor / SVR / XGBRegressor)
        rf_params: obsoleto, alias di `model_params` quando model_type="rf". Se sono passati entrambi, `model_params` ha la precedenza.
        """
        self.model_type = model_type
        # retrocompatibilità: rf_params nella versione precedente
        if rf_params is not None and model_params is None:
            model_params = rf_params
        self.model_params = model_params or {}
        self.temporal_params = temporal_params or {}
        self.spatial_params = spatial_params or {}
        self.kriging_params = kriging_params or {}
        self.lambda_grid = lambda_grid if lambda_grid is not None else np.linspace(0, 1, 11)
        self.cv_frac_stations = cv_frac_stations
        self.cv_seed = cv_seed
        self.use_km_projection = use_km_projection
        self.fixed_lambda = fixed_lambda

        self.rf_ = None
        self.cov_ = None
        self.kriging_ = None
        self.registry_ = None
        self.lambda_ = None
        self.diagnostics_ = {}

    def _fit_rf(self, X_obs, z_obs, n_predictors):
        """Costruisce e adatta il modello di larga scala (RF, SVR o XGBoost a seconda di self.model_type)
        """
        model = build_large_scale_model(self.model_type, self.model_params, n_predictors)

        max_rows = getattr(model, "_svr_max_train_rows", None)
        if max_rows is not None and len(X_obs) > max_rows:
            rng = np.random.default_rng(13)
            sub_idx = rng.choice(len(X_obs), size=max_rows, replace=False)
            print(f"[RFSTK] model_type='svr': training set ({len(X_obs)} righe) supera "
                f"max_train_rows={max_rows}. Adattamenti su un sottocampione casuale di "
                f"{max_rows} righe per contenere i tempi. "
                f"Passa model_params={{'max_train_rows': N}} per cambiare la soglia.")
            model.fit(X_obs[sub_idx], z_obs[sub_idx])
            return model

        model.fit(X_obs, z_obs)
        return model

    def fit(self, X, z, station_id, time, station_ids_unique, station_lat, station_lon):
        X = np.asarray(X, dtype=float)
        z = np.asarray(z, dtype=float)
        station_id = np.asarray(station_id)
        time = np.asarray(time, dtype=float)

        if self.use_km_projection:
            x_km, y_km, lat0, lon0 = latlon_to_km(station_lat, station_lon)
            self._proj_ref = (lat0, lon0)
            coords = np.column_stack([x_km, y_km])
        else:
            # Coordinate in gradi decimali (lat, lon), no proiezione
            self._proj_ref = None
            coords = np.column_stack([station_lat, station_lon])
        self.registry_ = StationRegistry(station_ids_unique, coords)
        station_idx_all = self.registry_.ids_to_indices(station_id)

        mask_obs = ~np.isnan(z)
        X_obs, z_obs = X[mask_obs], z[mask_obs]
        station_idx_obs, time_obs = station_idx_all[mask_obs], time[mask_obs]

        if self.fixed_lambda is not None:
            # lambda fisso: nessuna CV interna, nessun adattamento extra del modello di grande scala
            self.lambda_ = float(self.fixed_lambda)
            self.diagnostics_["cv_r2_by_lambda"] = None
            self.diagnostics_["cv_theta_s_km"] = None
            self.diagnostics_["cv_theta_t_days"] = None
        else:
            #divisione interna per la CV del moltiplicatore (leave-stations-out)
            stations_unique = np.unique(station_idx_obs)
            rng = np.random.default_rng(self.cv_seed)
            n_cv_test = max(1, int(round(len(stations_unique) * self.cv_frac_stations)))
            cv_test_stations = rng.choice(stations_unique, size=n_cv_test, replace=False)
            cv_test_mask = np.isin(station_idx_obs, cv_test_stations)
            cv_train_mask = ~cv_test_mask

            #RF
            rf_cv = self._fit_rf(X_obs[cv_train_mask], z_obs[cv_train_mask], X.shape[1])
            resid_cv_train = z_obs[cv_train_mask] - rf_cv.predict(X_obs[cv_train_mask])

            theta_t_cv, *_ = fit_marginal_temporal(resid_cv_train, station_idx_obs[cv_train_mask], time_obs[cv_train_mask],**self.temporal_params,)
            theta_s_cv, *_ = fit_marginal_spatial(resid_cv_train, station_idx_obs[cv_train_mask], time_obs[cv_train_mask],self.registry_, **self.spatial_params,)
            cov_cv = SeparableExponentialCovariance(total_var=float(np.var(resid_cv_train)),nugget=float(np.var(resid_cv_train)) * 0.1,theta_s=theta_s_cv, theta_t=theta_t_cv,)
            krig_cv = LocalSpatioTemporalKriging(cov_cv, **self.kriging_params).fit(resid_cv_train, station_idx_obs[cv_train_mask], time_obs[cv_train_mask], self.registry_)

            S_cv_test = rf_cv.predict(X_obs[cv_test_mask])
            resid_true_cv_test = z_obs[cv_test_mask] - S_cv_test
            resid_krig_cv_test, _ = krig_cv.predict(station_idx_obs[cv_test_mask], time_obs[cv_test_mask])

            best_lambda, best_r2 = 0.0, -np.inf
            r2_by_lambda = {}
            ss_tot = np.sum((resid_true_cv_test - resid_true_cv_test.mean()) ** 2)
            for lam in self.lambda_grid:
                pred = S_cv_test + lam * resid_krig_cv_test
                ss_res = np.sum((z_obs[cv_test_mask] - pred) ** 2)
                r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
                r2_by_lambda[float(lam)] = float(r2)
                if r2 > best_r2:
                    best_r2, best_lambda = r2, float(lam)

            self.lambda_ = best_lambda
            self.diagnostics_["cv_r2_by_lambda"] = r2_by_lambda
            self.diagnostics_["cv_theta_s_km"] = theta_s_cv
            self.diagnostics_["cv_theta_t_days"] = theta_t_cv

        # adattamento finale su tutti i dati di training disponibili
        self.rf_ = self._fit_rf(X_obs, z_obs, X.shape[1])
        residuals = z_obs - self.rf_.predict(X_obs)

        theta_t, lags_t, corr_t, n_t = fit_marginal_temporal(residuals, station_idx_obs, time_obs, **self.temporal_params)
        theta_s, centers_s, corr_s, n_s = fit_marginal_spatial(residuals, station_idx_obs, time_obs, self.registry_, **self.spatial_params)

        self.cov_ = SeparableExponentialCovariance(total_var=float(np.var(residuals)),nugget=float(np.var(residuals)) * 0.1,theta_s=theta_s, theta_t=theta_t,)
        self.kriging_ = LocalSpatioTemporalKriging(self.cov_, **self.kriging_params).fit(residuals, station_idx_obs, time_obs, self.registry_)

        self._residuals_train = residuals
        self._oob_score = _get_oob_score(self.rf_)
        self.diagnostics_["temporal_fit"] = dict(lags=lags_t, corr=corr_t, n_pairs=n_t)
        self.diagnostics_["spatial_fit"] = dict(centers_km=centers_s, corr=corr_s, n_pairs=n_s)

        return self

    def predict(self, X_new, station_id_new, time_new, return_variance=False, apply_lambda=True):
        if self.rf_ is None or self.kriging_ is None:
            raise RuntimeError("Il modello non è stato ancora allenato: chiama .fit()")

        X_new = np.asarray(X_new, dtype=float)
        station_id_new = np.asarray(station_id_new)
        time_new = np.asarray(time_new, dtype=float)
        station_idx_new = self.registry_.ids_to_indices(station_id_new)

        S_new = self.rf_.predict(X_new)
        resid_pred, resid_var = self.kriging_.predict(station_idx_new, time_new)

        lam = self.lambda_ if apply_lambda else 1.0
        z_pred = S_new + lam * resid_pred

        if return_variance:
            return z_pred, resid_var
        return z_pred

    def summary(self) -> dict:
        if self.cov_ is None:
            raise RuntimeError("Il modello non è stato ancora allenato: chiama .fit()")
        unit_s = "km" if self.use_km_projection else "gradi_decimali"
        return {"model_type": self.model_type,
            "oob_score_rf": self._oob_score,  # solo se model_type="rf"
            "unita' spaziali": unit_s,

            "total_var": self.cov_.total_var, "nugget": self.cov_.nugget,
            "partial_sill": self.cov_.partial_sill,
            f"theta_s_{unit_s}": self.cov_.theta_s, "theta_t_days": self.cov_.theta_t,
            
            "lambda_shrinkage": self.lambda_,
            "cv_r2_by_lambda": self.diagnostics_.get("cv_r2_by_lambda"),}

#Addenda
def train_test_split_by_station(station_id, test_size=0.2, seed=13):
    station_id = np.asarray(station_id)
    stations_unique = np.unique(station_id)
    rng = np.random.default_rng(seed)
    n_test = max(1, int(round(len(stations_unique) * test_size)))
    test_stations = rng.choice(stations_unique, size=n_test, replace=False)
    mask_test = np.isin(station_id, test_stations)
    mask_train = ~mask_test
    return mask_train, mask_test


def add_harmonic_time_features(time, periods):
    time = np.asarray(time, dtype=float)
    features = []
    for p in periods:
        angle = 2.0 * np.pi * time / p
        features.append(np.sin(angle))
        features.append(np.cos(angle))
    return np.column_stack(features)


def evaluate_predictions(y_true, y_pred, ignore_nan=True):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError("dimensioni incompatibili")
    mask = ~(np.isnan(y_true) | np.isnan(y_pred)) if ignore_nan else np.ones_like(y_true, dtype=bool)
    yt, yp = y_true[mask], y_pred[mask]
    n = len(yt)
    if n == 0:
        raise ValueError("nessun valore valido")
    errors = yt - yp
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    mae = float(np.mean(np.abs(errors)))
    ss_res = np.sum(errors ** 2)
    ss_tot = np.sum((yt - yt.mean()) ** 2)
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return {"rmse": rmse, "mae": mae, "r2": r2, "n_valutati": n}

In [3]:
#definizione dei dati
percorso = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/completi.xlsx'
c_staz = 'stazione'
c_data = 'data'
c_geo = ['comune','provincia','latitudine','longitudine']

tab = pd.read_excel(percorso)
tab = tab.sort_values([c_staz,c_data]).reset_index(drop=True)
seme = np.random.default_rng(13)

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=False).astype(float)
tab = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)


tab["time_num"] = (tab["data"] - tab["data"].min()).dt.days.astype(float)
ind_t = tab["time_num"].values
t_arm = add_harmonic_time_features(tab["time_num"].values, periods=[365.25, 7])

tab["vel10"] = np.sqrt(tab["u10_media"].to_numpy()**2 + tab["v10_media"].to_numpy()**2)
tab["vel100"] = np.sqrt(tab["u100_media"].to_numpy()**2 + tab["v100_media"].to_numpy()**2)

In [4]:
#costruzione della matrice dei dati selezionati 

c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','d2m_media','t2m_media','blh_media','bovini', 'ovini','suini', 'pollame','rh']
c_previsione = 'PM10_media'

c_misure.remove('u100_media')
c_misure.remove('v100_media')
c_misure.remove('u10_media')
c_misure.remove('v10_media')
c_misure = c_misure+["vel10", "vel100"]+col_mens

id_staz = tab["stazione"].values

X = np.column_stack([tab[c_misure].values,t_arm])
y_vera = tab[c_previsione].to_numpy()
coord_uniche = tab.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]

In [7]:
mask_train, mask_test = train_test_split_by_station(id_staz, test_size=0.2, seed=13)

model_xgb = RFSTK(model_type='xgboost',model_params=dict(n_estimators=400, max_depth=3),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_xgb.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
print(model_xgb.summary())  # sill, theta_s, theta_t, nugget, diagnostica bin
y_prev_xgb = model_xgb.predict(X[mask_test], id_staz[mask_test], ind_t[mask_test])


model_rf = RFSTK(model_type='rf',model_params=dict(n_estimators=400, min_samples_leaf=100),kriging_params=dict(max_neighbors=30*10, time_window=30.0), fixed_lambda=1., use_km_projection=True )
model_rf.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
print(model_rf.summary()) 
y_prev_rf = model_rf.predict(X[mask_test], id_staz[mask_test], ind_t[mask_test])


model_svm = RFSTK(model_type='svr',model_params=dict(C=0.1, epsilon=0.25, gamma="scale",max_train_rows=1826*10,),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_svm.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
print(model_svm.summary())
y_prev_svm = model_svm.predict(X[mask_test], id_staz[mask_test], ind_t[mask_test])



{'model_type': 'xgboost', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 92.49496241939327, 'nugget': 9.249496241939328, 'partial_sill': 83.24546617745395, 'theta_s_km': 109.46674267443818, 'theta_t_days': 2.879412190799829, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}
{'model_type': 'rf', 'oob_score_rf': 0.44800757452483464, "unita' spaziali": 'km', 'total_var': 71.97178682210897, 'nugget': 7.197178682210897, 'partial_sill': 64.77460813989808, 'theta_s_km': 80.19332883170608, 'theta_t_days': 2.3656087460178123, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}
[RFSTK] model_type='svr': training set (84391 righe) supera max_train_rows=18260. Adattamenti su un sottocampione casuale di 18260 righe per contenere i tempi. Passa model_params={'max_train_rows': N} per cambiare la soglia.
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 109.81039310921678, 'nugget': 10.981039310921679, 'partial_sill': 98.8293537982951, 'theta_s_km': 186.79609

In [8]:
metrics_xgb = evaluate_predictions(y_vera[mask_test], y_prev_xgb)
 
print(f"\nValutazione XGBoost su stazioni di test:")
print(f"  RMSE: {metrics_xgb['rmse']:.3f}")
print(f"  MAE:  {metrics_xgb['mae']:.3f}")
print(f"  R2:   {metrics_xgb['r2']:.3f}")
print(f"  (calcolate su {metrics_xgb['n_valutati']} osservazioni valide)")

metrics_rf = evaluate_predictions(y_vera[mask_test], y_prev_rf)
 
print(f"\nValutazione RF su stazioni di test:")
print(f"  RMSE: {metrics_rf['rmse']:.3f}")
print(f"  MAE:  {metrics_rf['mae']:.3f}")
print(f"  R2:   {metrics_rf['r2']:.3f}")
print(f"  (calcolate su {metrics_rf['n_valutati']} osservazioni valide)")

metrics_svm = evaluate_predictions(y_vera[mask_test], y_prev_svm)

print(f"\nValutazione SVM su stazioni di test:")
print(f"  RMSE: {metrics_svm['rmse']:.3f}")
print(f"  MAE:  {metrics_svm['mae']:.3f}")
print(f"  R2:   {metrics_svm['r2']:.3f}")
print(f"  (calcolate su {metrics_svm['n_valutati']} osservazioni valide)")


Valutazione XGBoost su stazioni di test:
  RMSE: 6.899
  MAE:  4.394
  R2:   0.675
  (calcolate su 21531 osservazioni valide)

Valutazione RF su stazioni di test:
  RMSE: 6.931
  MAE:  4.396
  R2:   0.672
  (calcolate su 21531 osservazioni valide)

Valutazione SVM su stazioni di test:
  RMSE: 6.843
  MAE:  4.358
  R2:   0.680
  (calcolate su 21531 osservazioni valide)


In [9]:
S_test_only = model_rf.rf_.predict(X[mask_test])
metrics_rf_only = evaluate_predictions(y_vera[mask_test], S_test_only)
print("Solo RF (larga scala):", metrics_rf_only)

metrics_xgb = evaluate_predictions(y_vera[mask_test], y_prev_xgb)
print("RFSTK completo XGBoost:", metrics_xgb)
metrics_rf = evaluate_predictions(y_vera[mask_test], y_prev_rf)
print("RFSTK completo RF:", metrics_rf)
metrics_svm = evaluate_predictions(y_vera[mask_test], y_prev_svm)
print("RFSTK completo SVM:", metrics_svm)

Solo RF (larga scala): {'rmse': 9.13749523212328, 'mae': 6.027057922860053, 'r2': 0.4300422995644688, 'n_valutati': 21531}
RFSTK completo XGBoost: {'rmse': 6.898673791646104, 'mae': 4.393829826636523, 'r2': 0.6751225872013419, 'n_valutati': 21531}
RFSTK completo RF: {'rmse': 6.930659553462505, 'mae': 4.3958370639213, 'r2': 0.6721030092684814, 'n_valutati': 21531}
RFSTK completo SVM: {'rmse': 6.842737871874949, 'mae': 4.358254333365356, 'r2': 0.6803695795746718, 'n_valutati': 21531}


In [16]:
# modello per pm2.5

c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','d2m_media','t2m_media','blh_media','bovini', 'ovini','suini', 'pollame','rh']
c_previsione = 'PM2o5_media'

c_misure.remove('u100_media')
c_misure.remove('v100_media')
c_misure.remove('u10_media')
c_misure.remove('v10_media')
c_misure = c_misure+["vel10", "vel100"]+col_mens


staz_valide = []
for staz in tab.drop_duplicates("stazione")["stazione"].values:
    if sum(tab.loc[tab.iloc[:,0]==staz,"PM2o5_righe"])>20000:
        staz_valide.append(staz)

print(len(staz_valide))

tab_mod = tab[tab.iloc[:,0].isin(staz_valide)]
X = None
y_vera = None


t_arm = add_harmonic_time_features(tab_mod["time_num"].values, periods=[365.25, 7])
ind_t = tab_mod["time_num"].values

id_staz = tab_mod["stazione"].values
X = np.column_stack([tab_mod[c_misure].values,t_arm])
#print(X.shape)
y_vera = np.log(tab_mod[c_previsione].to_numpy())
coord_uniche = tab_mod.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]

mask_train, mask_test = train_test_split_by_station(id_staz, test_size=0.2, seed=13)


model_svm = RFSTK(model_type='svr',model_params=dict(C=0.1, epsilon=0.2, gamma="scale",max_train_rows=1826*10,),kriging_params=dict(max_neighbors=40*10, time_window=30.0), fixed_lambda=1., use_km_projection=True )
model_svm.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
print(model_svm.summary())
y_prev_svm = model_svm.predict(X[mask_test], id_staz[mask_test], ind_t[mask_test])


metrics_svm = evaluate_predictions(y_vera[mask_test], y_prev_svm)

print(f"\nValutazione SVM su stazioni di test:")
print(f"  RMSE: {metrics_svm['rmse']:.3f}")
print(f"  MAE:  {metrics_svm['mae']:.3f}")
print(f"  R2:   {metrics_svm['r2']:.3f}")
print(f"  (calcolate su {metrics_svm['n_valutati']} osservazioni valide)")

model_rf = RFSTK(model_type='rf',model_params=dict(n_estimators=400, min_samples_leaf=100),kriging_params=dict(max_neighbors=30*10, time_window=30.0), fixed_lambda=1., use_km_projection=True )
model_rf.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
S_test_only = model_rf.rf_.predict(X[mask_test])
metrics_rf_only = evaluate_predictions(y_vera[mask_test], S_test_only)
print("Solo RF (larga scala):", metrics_rf_only)
metrics_svm = evaluate_predictions(y_vera[mask_test], y_prev_svm)
print("RFSTK completo SVM:", metrics_svm)

35
[RFSTK] model_type='svr': training set (45634 righe) supera max_train_rows=18260. Adattamenti su un sottocampione casuale di 18260 righe per contenere i tempi. Passa model_params={'max_train_rows': N} per cambiare la soglia.
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 0.15913886581771716, 'nugget': 0.015913886581771716, 'partial_sill': 0.14322497923594546, 'theta_s_km': 72.03637728349992, 'theta_t_days': 4.1327372118854075, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test:
  RMSE: 0.341
  MAE:  0.251
  R2:   0.587
  (calcolate su 10971 osservazioni valide)
Solo RF (larga scala): {'rmse': 0.4211166567451965, 'mae': 0.32420952579542384, 'r2': 0.37138872633981623, 'n_valutati': 10971}
RFSTK completo SVM: {'rmse': 0.3414198550080508, 'mae': 0.2507170166383923, 'r2': 0.5868051754389041, 'n_valutati': 10971}


In [21]:
# modello per nox, risolvere la stagionalità

c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','d2m_media','t2m_media','blh_media','bovini', 'ovini','suini', 'pollame','rh']
c_previsione = 'NOX_media'

c_misure.remove('u100_media')
c_misure.remove('v100_media')
c_misure.remove('u10_media')
c_misure.remove('v10_media')


staz_valide = []
for staz in tab.drop_duplicates("stazione")["stazione"].values:
    if sum(tab.loc[tab.iloc[:,0]==staz,"NOX_righe"])>20000:
        staz_valide.append(staz)

print(len(staz_valide))

tab_mod = tab[tab.iloc[:,0].isin(staz_valide)]
X = None
y_vera = None


tab_mod['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab_mod['mese'], prefix='mese',drop_first=False).astype(float)
tab_mod = pd.concat([tab_mod,regr_mens],axis=1)
col_mens = list(regr_mens.columns)

c_misure = c_misure+["vel10", "vel100"]+col_mens

t_arm = add_harmonic_time_features(tab_mod["time_num"].values, periods=[365.25, 7])
ind_t = tab_mod["time_num"].values

id_staz = tab_mod["stazione"].values
X = np.column_stack([tab_mod[c_misure].values,t_arm])
#print(X.shape)
y_vera = (tab_mod[c_previsione].to_numpy())
coord_uniche = tab_mod.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]

mask_train, mask_test = train_test_split_by_station(id_staz, test_size=0.2, seed=13)

model_xgb = RFSTK(model_type='xgboost',model_params=dict(n_estimators=50, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_xgb.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
print(model_xgb.summary()) 
y_prev_xgb = model_xgb.predict(X[mask_test], id_staz[mask_test], ind_t[mask_test])

metrics_xgb = evaluate_predictions(y_vera[mask_test], y_prev_xgb)

print(f"\nValutazione SVM su stazioni di test:")
print(f"  RMSE: {metrics_xgb['rmse']:.3f}")
print(f"  MAE:  {metrics_xgb['mae']:.3f}")
print(f"  R2:   {metrics_xgb['r2']:.3f}")
print(f"  (calcolate su {metrics_xgb['n_valutati']} osservazioni valide)")


model_rf = RFSTK(model_type='rf',model_params=dict(n_estimators=50, min_samples_leaf=2),kriging_params=dict(max_neighbors=30*10, time_window=7.0), fixed_lambda=1., use_km_projection=True )
model_rf.fit(X[mask_train], y_vera[mask_train], id_staz[mask_train], ind_t[mask_train],coord_uniche["stazione"].to_numpy(), coord_uniche["latitudine"].to_numpy(),coord_uniche["longitudine"].to_numpy(),)
S_test_only = model_rf.rf_.predict(X[mask_test])
metrics_rf_only = evaluate_predictions(y_vera[mask_test], S_test_only)
print("Solo RF (larga scala):", metrics_rf_only)
metrics_svm = evaluate_predictions(y_vera[mask_test], y_prev_xgb)
print("RFSTK completo xgboost:", metrics_xgb)

61
{'model_type': 'xgboost', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 281.4154510717421, 'nugget': 28.14154510717421, 'partial_sill': 253.2739059645679, 'theta_s_km': 7.983683006886155, 'theta_t_days': 27.4673410105939, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test:
  RMSE: 11.213
  MAE:  8.631
  R2:   0.286
  (calcolate su 20013 osservazioni valide)
Solo RF (larga scala): {'rmse': 12.66540559247007, 'mae': 8.861044745543811, 'r2': 0.08928311138028655, 'n_valutati': 20013}
RFSTK completo xgboost: {'rmse': 11.212722571442411, 'mae': 8.631006592867676, 'r2': 0.28621512687108486, 'n_valutati': 20013}


In [ ]:
# modello provinciale pm10

percorso_prov = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/id stazioni.xlsx'

tab_staz = pd.read_excel(percorso_prov)
tab_staz = tab_staz.sort_values("id staz").reset_index(drop=True)

staz_ba = tab_staz["nome staz"][:15].values
staz_bt = tab_staz["nome staz"][15:19].values
staz_br = tab_staz["nome staz"][19:35].values
staz_fg = tab_staz["nome staz"][35:42].values
staz_le = tab_staz["nome staz"][42:52].values
staz_ta = tab_staz["nome staz"][52:].values

tab_ba = tab[tab["stazione"].isin(staz_ba)]
tab_bt = tab[tab["stazione"].isin(staz_bt)]
tab_br = tab[tab["stazione"].isin(staz_br)]
tab_fg = tab[tab["stazione"].isin(staz_fg)]
tab_le = tab[tab["stazione"].isin(staz_le)]
tab_ta = tab[tab["stazione"].isin(staz_ta)]

id_ba = tab_ba["stazione"].values
id_bt = tab_bt["stazione"].values
id_br = tab_br["stazione"].values
id_fg = tab_fg["stazione"].values
id_le = tab_le["stazione"].values
id_ta = tab_ta["stazione"].values

tab_ba["time_num"] = (tab_ba["data"] - tab_ba["data"].min()).dt.days.astype(float)
ind_t_ba = tab_ba["time_num"].values
t_arm_ba = add_harmonic_time_features(tab_ba["time_num"].values, periods=[365.25, 7])
tab_bt["time_num"] = (tab_bt["data"] - tab_bt["data"].min()).dt.days.astype(float)
ind_t_bt = tab_bt["time_num"].values
t_arm_bt = add_harmonic_time_features(tab_bt["time_num"].values, periods=[365.25, 7])
tab_br["time_num"] = (tab_br["data"] - tab_br["data"].min()).dt.days.astype(float)
ind_t_br = tab_br["time_num"].values
t_arm_br = add_harmonic_time_features(tab_br["time_num"].values, periods=[365.25, 7])
tab_fg["time_num"] = (tab_fg["data"] - tab_fg["data"].min()).dt.days.astype(float)
ind_t_fg = tab_fg["time_num"].values
t_arm_fg = add_harmonic_time_features(tab_fg["time_num"].values, periods=[365.25, 7])
tab_le["time_num"] = (tab_le["data"] - tab_le["data"].min()).dt.days.astype(float)
ind_t_le = tab_le["time_num"].values
t_arm_le = add_harmonic_time_features(tab_le["time_num"].values, periods=[365.25, 7])
tab_ta["time_num"] = (tab_ta["data"] - tab_ta["data"].min()).dt.days.astype(float)
ind_t_ta = tab_ta["time_num"].values
t_arm_ta = add_harmonic_time_features(tab_ta["time_num"].values, periods=[365.25, 7])

X_ba = np.column_stack([tab_ba[c_misure].values,t_arm_ba])
y_ba = tab_ba[c_previsione].to_numpy()
coord_ba = tab_ba.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_bt = np.column_stack([tab_bt[c_misure].values,t_arm_bt])
y_bt = tab_bt[c_previsione].to_numpy()
coord_bt = tab_bt.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_br = np.column_stack([tab_br[c_misure].values,t_arm_br])
y_br = tab_br[c_previsione].to_numpy()
coord_br = tab_br.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_fg = np.column_stack([tab_fg[c_misure].values,t_arm_fg])
y_fg = tab_fg[c_previsione].to_numpy()
coord_fg = tab_fg.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_le = np.column_stack([tab_le[c_misure].values,t_arm_le])
y_le = tab_le[c_previsione].to_numpy()
coord_le = tab_le.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_ta = np.column_stack([tab_ta[c_misure].values,t_arm_ta])
y_ta = tab_ta[c_previsione].to_numpy()
coord_ta = tab_ta.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]

In [ ]:
mask_train_ba, mask_test_ba = train_test_split_by_station(id_ba, test_size=0.2, seed=13)

model_ba = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_ba)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_ba.fit(X_ba[mask_train_ba], y_ba[mask_train_ba], id_ba[mask_train_ba], ind_t_ba[mask_train_ba],coord_ba["stazione"].to_numpy(), coord_ba["latitudine"].to_numpy(),coord_ba["longitudine"].to_numpy(),)
print(model_ba.summary()) 
y_prev_ba = model_ba.predict(X_ba[mask_test_ba], id_ba[mask_test_ba], ind_t_ba[mask_test_ba])

metrics_ba = evaluate_predictions(y_ba[mask_test_ba], y_prev_ba)

print(f"\nValutazione SVM su stazioni di test BA:")
print(f"  RMSE: {metrics_ba['rmse']:.3f}")
print(f"  MAE:  {metrics_ba['mae']:.3f}")
print(f"  R2:   {metrics_ba['r2']:.3f}")
print(f"  (calcolate su {metrics_ba['n_valutati']} osservazioni valide)")

mask_train_bt, mask_test_bt = train_test_split_by_station(id_bt, test_size=0.2, seed=13)

model_bt = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_bt)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_bt.fit(X_bt[mask_train_bt], y_bt[mask_train_bt], id_bt[mask_train_bt], ind_t_bt[mask_train_bt],coord_bt["stazione"].to_numpy(), coord_bt["latitudine"].to_numpy(),coord_bt["longitudine"].to_numpy(),)
print(model_bt.summary()) 
y_prev_bt = model_bt.predict(X_bt[mask_test_bt], id_bt[mask_test_bt], ind_t_bt[mask_test_bt])

metrics_bt = evaluate_predictions(y_bt[mask_test_bt], y_prev_bt)

print(f"\nValutazione SVM su stazioni di test BT:")
print(f"  RMSE: {metrics_bt['rmse']:.3f}")
print(f"  MAE:  {metrics_bt['mae']:.3f}")
print(f"  R2:   {metrics_bt['r2']:.3f}")
print(f"  (calcolate su {metrics_bt['n_valutati']} osservazioni valide)")

mask_train_br, mask_test_br = train_test_split_by_station(id_br, test_size=0.2, seed=13)

model_br = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_br)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_br.fit(X_br[mask_train_br], y_br[mask_train_br], id_br[mask_train_br], ind_t_br[mask_train_br],coord_br["stazione"].to_numpy(), coord_br["latitudine"].to_numpy(),coord_br["longitudine"].to_numpy(),)
print(model_br.summary()) 
y_prev_br = model_br.predict(X_br[mask_test_br], id_br[mask_test_br], ind_t_br[mask_test_br])

metrics_br = evaluate_predictions(y_br[mask_test_br], y_prev_br)

print(f"\nValutazione SVM su stazioni di test BR:")
print(f"  RMSE: {metrics_br['rmse']:.3f}")
print(f"  MAE:  {metrics_br['mae']:.3f}")
print(f"  R2:   {metrics_br['r2']:.3f}")
print(f"  (calcolate su {metrics_br['n_valutati']} osservazioni valide)")

mask_train_fg, mask_test_fg = train_test_split_by_station(id_fg, test_size=0.2, seed=13)

model_fg = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_fg)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_fg.fit(X_fg[mask_train_fg], y_fg[mask_train_fg], id_fg[mask_train_fg], ind_t_fg[mask_train_fg],coord_fg["stazione"].to_numpy(), coord_fg["latitudine"].to_numpy(),coord_fg["longitudine"].to_numpy(),)
print(model_fg.summary()) 
y_prev_fg = model_fg.predict(X_fg[mask_test_fg], id_fg[mask_test_fg], ind_t_fg[mask_test_fg])

metrics_fg = evaluate_predictions(y_fg[mask_test_fg], y_prev_fg)

print(f"\nValutazione SVM su stazioni di test FG:")
print(f"  RMSE: {metrics_fg['rmse']:.3f}")
print(f"  MAE:  {metrics_fg['mae']:.3f}")
print(f"  R2:   {metrics_fg['r2']:.3f}")
print(f"  (calcolate su {metrics_fg['n_valutati']} osservazioni valide)")

mask_train_le, mask_test_le = train_test_split_by_station(id_le, test_size=0.2, seed=13)

model_le = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_le)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_le.fit(X_le[mask_train_le], y_le[mask_train_le], id_le[mask_train_le], ind_t_le[mask_train_le],coord_le["stazione"].to_numpy(), coord_le["latitudine"].to_numpy(),coord_le["longitudine"].to_numpy(),)
print(model_le.summary()) 
y_prev_le = model_le.predict(X_le[mask_test_le], id_le[mask_test_le], ind_t_le[mask_test_le])

metrics_le = evaluate_predictions(y_le[mask_test_le], y_prev_le)

print(f"\nValutazione SVM su stazioni di test LE:")
print(f"  RMSE: {metrics_le['rmse']:.3f}")
print(f"  MAE:  {metrics_le['mae']:.3f}")
print(f"  R2:   {metrics_le['r2']:.3f}")
print(f"  (calcolate su {metrics_le['n_valutati']} osservazioni valide)")

mask_train_ta, mask_test_ta = train_test_split_by_station(id_ta, test_size=0.2, seed=13)

model_ta = RFSTK(model_type='svr',model_params=dict(C=20., epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_ta)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_ta.fit(X_ta[mask_train_ta], y_ta[mask_train_ta], id_ta[mask_train_ta], ind_t_ta[mask_train_ta],coord_ta["stazione"].to_numpy(), coord_ta["latitudine"].to_numpy(),coord_ta["longitudine"].to_numpy(),)
print(model_ta.summary()) 
y_prev_ta = model_ta.predict(X_ta[mask_test_ta], id_ta[mask_test_ta], ind_t_ta[mask_test_ta])

metrics_ta = evaluate_predictions(y_ta[mask_test_ta], y_prev_ta)

print(f"\nValutazione SVM su stazioni di test TA:")
print(f"  RMSE: {metrics_ta['rmse']:.3f}")
print(f"  MAE:  {metrics_ta['mae']:.3f}")
print(f"  R2:   {metrics_ta['r2']:.3f}")
print(f"  (calcolate su {metrics_ta['n_valutati']} osservazioni valide)")

{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 109.66715793761892, 'nugget': 10.966715793761892, 'partial_sill': 98.70044214385703, 'theta_s_km': 34.05871735640501, 'theta_t_days': 3.4736828913261686, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test BA:
  RMSE: 6.061
  MAE:  4.243
  R2:   0.649
  (calcolate su 4983 osservazioni valide)
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 195.6653480620143, 'nugget': 19.56653480620143, 'partial_sill': 176.09881325581284, 'theta_s_km': 12.801077301844595, 'theta_t_days': 4.778636874615208, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test BT:
  RMSE: 6.657
  MAE:  4.398
  R2:   0.673
  (calcolate su 683 osservazioni valide)
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 142.62394734821186, 'nugget': 14.262394734821186, 'partial_sill': 128.36155261339067, 'theta_s_km'

In [ ]:
# modello provinciale per pm2.5

percorso_prov = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/id stazioni.xlsx'
c_previsione = 'PM2o5_media'

staz_valide = []
for staz in tab.drop_duplicates("stazione")["stazione"].values:
    if sum(tab.loc[tab.iloc[:,0]==staz,"PM2o5_righe"])>700:
        staz_valide.append(staz)

print(len(staz_valide))

tab_mod = tab[tab.iloc[:,0].isin(staz_valide)]

tab_staz = pd.read_excel(percorso_prov)
tab_staz = tab_staz.sort_values("id staz").reset_index(drop=True)

staz_ba = tab_staz["nome staz"][:15].values
staz_bt = tab_staz["nome staz"][15:19].values
staz_br = tab_staz["nome staz"][19:35].values
staz_fg = tab_staz["nome staz"][35:42].values
staz_le = tab_staz["nome staz"][42:52].values
staz_ta = tab_staz["nome staz"][52:].values

tab_ba = tab_mod[tab_mod["stazione"].isin(staz_ba)]
tab_bt = tab_mod[tab_mod["stazione"].isin(staz_bt)]
tab_br = tab_mod[tab_mod["stazione"].isin(staz_br)]
tab_fg = tab_mod[tab_mod["stazione"].isin(staz_fg)]
tab_le = tab_mod[tab_mod["stazione"].isin(staz_le)]
tab_ta = tab_mod[tab_mod["stazione"].isin(staz_ta)]

id_ba = tab_ba["stazione"].values
id_bt = tab_bt["stazione"].values
id_br = tab_br["stazione"].values
id_fg = tab_fg["stazione"].values
id_le = tab_le["stazione"].values
id_ta = tab_ta["stazione"].values

tab_ba["time_num"] = (tab_ba["data"] - tab_ba["data"].min()).dt.days.astype(float)
ind_t_ba = tab_ba["time_num"].values
t_arm_ba = add_harmonic_time_features(tab_ba["time_num"].values, periods=[365.25, 7])
tab_bt["time_num"] = (tab_bt["data"] - tab_bt["data"].min()).dt.days.astype(float)
ind_t_bt = tab_bt["time_num"].values
t_arm_bt = add_harmonic_time_features(tab_bt["time_num"].values, periods=[365.25, 7])
tab_br["time_num"] = (tab_br["data"] - tab_br["data"].min()).dt.days.astype(float)
ind_t_br = tab_br["time_num"].values
t_arm_br = add_harmonic_time_features(tab_br["time_num"].values, periods=[365.25, 7])
tab_fg["time_num"] = (tab_fg["data"] - tab_fg["data"].min()).dt.days.astype(float)
ind_t_fg = tab_fg["time_num"].values
t_arm_fg = add_harmonic_time_features(tab_fg["time_num"].values, periods=[365.25, 7])
tab_le["time_num"] = (tab_le["data"] - tab_le["data"].min()).dt.days.astype(float)
ind_t_le = tab_le["time_num"].values
t_arm_le = add_harmonic_time_features(tab_le["time_num"].values, periods=[365.25, 7])
tab_ta["time_num"] = (tab_ta["data"] - tab_ta["data"].min()).dt.days.astype(float)
ind_t_ta = tab_ta["time_num"].values
t_arm_ta = add_harmonic_time_features(tab_ta["time_num"].values, periods=[365.25, 7])

X_ba = np.column_stack([tab_ba[c_misure].values,t_arm_ba])
y_ba = tab_ba[c_previsione].to_numpy()
coord_ba = tab_ba.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_bt = np.column_stack([tab_bt[c_misure].values,t_arm_bt])
y_bt = tab_bt[c_previsione].to_numpy()
coord_bt = tab_bt.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_br = np.column_stack([tab_br[c_misure].values,t_arm_br])
y_br = tab_br[c_previsione].to_numpy()
coord_br = tab_br.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_fg = np.column_stack([tab_fg[c_misure].values,t_arm_fg])
y_fg = tab_fg[c_previsione].to_numpy()
coord_fg = tab_fg.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_le = np.column_stack([tab_le[c_misure].values,t_arm_le])
y_le = tab_le[c_previsione].to_numpy()
coord_le = tab_le.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_ta = np.column_stack([tab_ta[c_misure].values,t_arm_ta])
y_ta = tab_ta[c_previsione].to_numpy()
coord_ta = tab_ta.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]

mask_train_ba, mask_test_ba = train_test_split_by_station(id_ba, test_size=0.2, seed=13)

model_ba = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_ba)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_ba.fit(X_ba[mask_train_ba], y_ba[mask_train_ba], id_ba[mask_train_ba], ind_t_ba[mask_train_ba],coord_ba["stazione"].to_numpy(), coord_ba["latitudine"].to_numpy(),coord_ba["longitudine"].to_numpy(),)
print(model_ba.summary()) 
y_prev_ba = model_ba.predict(X_ba[mask_test_ba], id_ba[mask_test_ba], ind_t_ba[mask_test_ba])

metrics_ba = evaluate_predictions(y_ba[mask_test_ba], y_prev_ba)

print(f"\nValutazione SVM su stazioni di test BA:")
print(f"  RMSE: {metrics_ba['rmse']:.3f}")
print(f"  MAE:  {metrics_ba['mae']:.3f}")
print(f"  R2:   {metrics_ba['r2']:.3f}")
print(f"  (calcolate su {metrics_ba['n_valutati']} osservazioni valide)")

mask_train_bt, mask_test_bt = train_test_split_by_station(id_bt, test_size=0.2, seed=13)

model_bt = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_bt)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_bt.fit(X_bt[mask_train_bt], y_bt[mask_train_bt], id_bt[mask_train_bt], ind_t_bt[mask_train_bt],coord_bt["stazione"].to_numpy(), coord_bt["latitudine"].to_numpy(),coord_bt["longitudine"].to_numpy(),)
print(model_bt.summary()) 
y_prev_bt = model_bt.predict(X_bt[mask_test_bt], id_bt[mask_test_bt], ind_t_bt[mask_test_bt])

metrics_bt = evaluate_predictions(y_bt[mask_test_bt], y_prev_bt)

print(f"\nValutazione SVM su stazioni di test BT:")
print(f"  RMSE: {metrics_bt['rmse']:.3f}")
print(f"  MAE:  {metrics_bt['mae']:.3f}")
print(f"  R2:   {metrics_bt['r2']:.3f}")
print(f"  (calcolate su {metrics_bt['n_valutati']} osservazioni valide)")

mask_train_br, mask_test_br = train_test_split_by_station(id_br, test_size=0.2, seed=13)

model_br = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_br)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_br.fit(X_br[mask_train_br], y_br[mask_train_br], id_br[mask_train_br], ind_t_br[mask_train_br],coord_br["stazione"].to_numpy(), coord_br["latitudine"].to_numpy(),coord_br["longitudine"].to_numpy(),)
print(model_br.summary()) 
y_prev_br = model_br.predict(X_br[mask_test_br], id_br[mask_test_br], ind_t_br[mask_test_br])

metrics_br = evaluate_predictions(y_br[mask_test_br], y_prev_br)

print(f"\nValutazione SVM su stazioni di test BR:")
print(f"  RMSE: {metrics_br['rmse']:.3f}")
print(f"  MAE:  {metrics_br['mae']:.3f}")
print(f"  R2:   {metrics_br['r2']:.3f}")
print(f"  (calcolate su {metrics_br['n_valutati']} osservazioni valide)")

mask_train_fg, mask_test_fg = train_test_split_by_station(id_fg, test_size=0.2, seed=13)

model_fg = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_fg)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_fg.fit(X_fg[mask_train_fg], y_fg[mask_train_fg], id_fg[mask_train_fg], ind_t_fg[mask_train_fg],coord_fg["stazione"].to_numpy(), coord_fg["latitudine"].to_numpy(),coord_fg["longitudine"].to_numpy(),)
print(model_fg.summary()) 
y_prev_fg = model_fg.predict(X_fg[mask_test_fg], id_fg[mask_test_fg], ind_t_fg[mask_test_fg])

metrics_fg = evaluate_predictions(y_fg[mask_test_fg], y_prev_fg)

print(f"\nValutazione SVM su stazioni di test FG:")
print(f"  RMSE: {metrics_fg['rmse']:.3f}")
print(f"  MAE:  {metrics_fg['mae']:.3f}")
print(f"  R2:   {metrics_fg['r2']:.3f}")
print(f"  (calcolate su {metrics_fg['n_valutati']} osservazioni valide)")

mask_train_le, mask_test_le = train_test_split_by_station(id_le, test_size=0.2, seed=13)

model_le = RFSTK(model_type='svr',model_params=dict(C=0.01, epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_le)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_le.fit(X_le[mask_train_le], y_le[mask_train_le], id_le[mask_train_le], ind_t_le[mask_train_le],coord_le["stazione"].to_numpy(), coord_le["latitudine"].to_numpy(),coord_le["longitudine"].to_numpy(),)
print(model_le.summary()) 
y_prev_le = model_le.predict(X_le[mask_test_le], id_le[mask_test_le], ind_t_le[mask_test_le])

metrics_le = evaluate_predictions(y_le[mask_test_le], y_prev_le)

print(f"\nValutazione SVM su stazioni di test LE:")
print(f"  RMSE: {metrics_le['rmse']:.3f}")
print(f"  MAE:  {metrics_le['mae']:.3f}")
print(f"  R2:   {metrics_le['r2']:.3f}")
print(f"  (calcolate su {metrics_le['n_valutati']} osservazioni valide)")

mask_train_ta, mask_test_ta = train_test_split_by_station(id_ta, test_size=0.2, seed=13)

model_ta = RFSTK(model_type='svr',model_params=dict(C=20., epsilon=0.01, gamma="scale",max_train_rows=1826*int(len(id_ta)*0.6),),kriging_params=dict(max_neighbors=30*10, time_window=30.0),fixed_lambda=1., use_km_projection=True )
model_ta.fit(X_ta[mask_train_ta], y_ta[mask_train_ta], id_ta[mask_train_ta], ind_t_ta[mask_train_ta],coord_ta["stazione"].to_numpy(), coord_ta["latitudine"].to_numpy(),coord_ta["longitudine"].to_numpy(),)
print(model_ta.summary()) 
y_prev_ta = model_ta.predict(X_ta[mask_test_ta], id_ta[mask_test_ta], ind_t_ta[mask_test_ta])

metrics_ta = evaluate_predictions(y_ta[mask_test_ta], y_prev_ta)

print(f"\nValutazione SVM su stazioni di test TA:")
print(f"  RMSE: {metrics_ta['rmse']:.3f}")
print(f"  MAE:  {metrics_ta['mae']:.3f}")
print(f"  R2:   {metrics_ta['r2']:.3f}")
print(f"  (calcolate su {metrics_ta['n_valutati']} osservazioni valide)")

37
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 29.343619095168453, 'nugget': 2.9343619095168454, 'partial_sill': 26.409257185651608, 'theta_s_km': 77.49614076403148, 'theta_t_days': 4.052021947318017, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test BA:
  RMSE: 3.125
  MAE:  2.403
  R2:   0.677
  (calcolate su 3502 osservazioni valide)
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 56.723586912280254, 'nugget': 5.672358691228026, 'partial_sill': 51.05122822105223, 'theta_s_km': 15.797208382779004, 'theta_t_days': 2.930153857655046, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test BT:
  RMSE: 2.723
  MAE:  1.563
  R2:   0.823
  (calcolate su 677 osservazioni valide)
{'model_type': 'svr', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 170.64629767032247, 'nugget': 17.064629767032248, 'partial_sill': 153.58166790329022, 'theta_s_

In [18]:
# modello provinciale per nox

percorso_prov = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/id stazioni.xlsx'
c_previsione = 'NOX_media'

staz_valide = []
for staz in tab.drop_duplicates("stazione")["stazione"].values:
    if sum(tab.loc[tab.iloc[:,0]==staz,"NOX_righe"])>700:
        staz_valide.append(staz)

print(len(staz_valide))

tab_mod = tab[tab.iloc[:,0].isin(staz_valide)]

tab_staz = pd.read_excel(percorso_prov)
tab_staz = tab_staz.sort_values("id staz").reset_index(drop=True)

staz_ba = tab_staz["nome staz"][:15].values
staz_bt = tab_staz["nome staz"][15:19].values
staz_br = tab_staz["nome staz"][19:35].values
staz_fg = tab_staz["nome staz"][35:42].values
staz_le = tab_staz["nome staz"][42:52].values
staz_ta = tab_staz["nome staz"][52:].values

tab_ba = tab_mod[tab_mod["stazione"].isin(staz_ba)]
tab_bt = tab_mod[tab_mod["stazione"].isin(staz_bt)]
tab_br = tab_mod[tab_mod["stazione"].isin(staz_br)]
tab_fg = tab_mod[tab_mod["stazione"].isin(staz_fg)]
tab_le = tab_mod[tab_mod["stazione"].isin(staz_le)]
tab_ta = tab_mod[tab_mod["stazione"].isin(staz_ta)]

id_ba = tab_ba["stazione"].values
id_bt = tab_bt["stazione"].values
id_br = tab_br["stazione"].values
id_fg = tab_fg["stazione"].values
id_le = tab_le["stazione"].values
id_ta = tab_ta["stazione"].values

tab_ba["time_num"] = (tab_ba["data"] - tab_ba["data"].min()).dt.days.astype(float)
ind_t_ba = tab_ba["time_num"].values
t_arm_ba = add_harmonic_time_features(tab_ba["time_num"].values, periods=[365.25, 7])
tab_bt["time_num"] = (tab_bt["data"] - tab_bt["data"].min()).dt.days.astype(float)
ind_t_bt = tab_bt["time_num"].values
t_arm_bt = add_harmonic_time_features(tab_bt["time_num"].values, periods=[365.25, 7])
tab_br["time_num"] = (tab_br["data"] - tab_br["data"].min()).dt.days.astype(float)
ind_t_br = tab_br["time_num"].values
t_arm_br = add_harmonic_time_features(tab_br["time_num"].values, periods=[365.25, 7])
tab_fg["time_num"] = (tab_fg["data"] - tab_fg["data"].min()).dt.days.astype(float)
ind_t_fg = tab_fg["time_num"].values
t_arm_fg = add_harmonic_time_features(tab_fg["time_num"].values, periods=[365.25, 7])
tab_le["time_num"] = (tab_le["data"] - tab_le["data"].min()).dt.days.astype(float)
ind_t_le = tab_le["time_num"].values
t_arm_le = add_harmonic_time_features(tab_le["time_num"].values, periods=[365.25, 7])
tab_ta["time_num"] = (tab_ta["data"] - tab_ta["data"].min()).dt.days.astype(float)
ind_t_ta = tab_ta["time_num"].values
t_arm_ta = add_harmonic_time_features(tab_ta["time_num"].values, periods=[365.25, 7])

X_ba = np.column_stack([tab_ba[c_misure].values,t_arm_ba])
y_ba = tab_ba[c_previsione].to_numpy()
coord_ba = tab_ba.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_bt = np.column_stack([tab_bt[c_misure].values,t_arm_bt])
y_bt = tab_bt[c_previsione].to_numpy()
coord_bt = tab_bt.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_br = np.column_stack([tab_br[c_misure].values,t_arm_br])
y_br = tab_br[c_previsione].to_numpy()
coord_br = tab_br.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_fg = np.column_stack([tab_fg[c_misure].values,t_arm_fg])
y_fg = tab_fg[c_previsione].to_numpy()
coord_fg = tab_fg.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_le = np.column_stack([tab_le[c_misure].values,t_arm_le])
y_le = tab_le[c_previsione].to_numpy()
coord_le = tab_le.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]
X_ta = np.column_stack([tab_ta[c_misure].values,t_arm_ta])
y_ta = tab_ta[c_previsione].to_numpy()
coord_ta = tab_ta.drop_duplicates("stazione")[["stazione", "latitudine", "longitudine"]]

mask_train_ba, mask_test_ba = train_test_split_by_station(id_ba, test_size=0.2, seed=13)

model_ba = RFSTK(model_type='xgboost',model_params=dict(n_estimators=50, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_ba.fit(X_ba[mask_train_ba], y_ba[mask_train_ba], id_ba[mask_train_ba], ind_t_ba[mask_train_ba],coord_ba["stazione"].to_numpy(), coord_ba["latitudine"].to_numpy(),coord_ba["longitudine"].to_numpy(),)
print(model_ba.summary()) 
y_prev_ba = model_ba.predict(X_ba[mask_test_ba], id_ba[mask_test_ba], ind_t_ba[mask_test_ba])

metrics_ba = evaluate_predictions(y_ba[mask_test_ba], y_prev_ba)

print(f"\nValutazione SVM su stazioni di test BA:")
print(f"  RMSE: {metrics_ba['rmse']:.3f}")
print(f"  MAE:  {metrics_ba['mae']:.3f}")
print(f"  R2:   {metrics_ba['r2']:.3f}")
print(f"  (calcolate su {metrics_ba['n_valutati']} osservazioni valide)")

mask_train_bt, mask_test_bt = train_test_split_by_station(id_bt, test_size=0.2, seed=13)

model_bt = RFSTK(model_type='xgboost',model_params=dict(n_estimators=50, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_bt.fit(X_bt[mask_train_bt], y_bt[mask_train_bt], id_bt[mask_train_bt], ind_t_bt[mask_train_bt],coord_bt["stazione"].to_numpy(), coord_bt["latitudine"].to_numpy(),coord_bt["longitudine"].to_numpy(),)
print(model_bt.summary()) 
y_prev_bt = model_bt.predict(X_bt[mask_test_bt], id_bt[mask_test_bt], ind_t_bt[mask_test_bt])


metrics_bt = evaluate_predictions(y_bt[mask_test_bt], y_prev_bt)

print(f"\nValutazione SVM su stazioni di test BT:")
print(f"  RMSE: {metrics_bt['rmse']:.3f}")
print(f"  MAE:  {metrics_bt['mae']:.3f}")
print(f"  R2:   {metrics_bt['r2']:.3f}")
print(f"  (calcolate su {metrics_bt['n_valutati']} osservazioni valide)")

mask_train_br, mask_test_br = train_test_split_by_station(id_br, test_size=0.2, seed=13)

model_br = RFSTK(model_type='xgboost',model_params=dict(n_estimators=25, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_br.fit(X_br[mask_train_br], y_br[mask_train_br], id_br[mask_train_br], ind_t_br[mask_train_br],coord_br["stazione"].to_numpy(), coord_br["latitudine"].to_numpy(),coord_br["longitudine"].to_numpy(),)
print(model_br.summary()) 
y_prev_br = model_br.predict(X_br[mask_test_br], id_br[mask_test_br], ind_t_br[mask_test_br])


metrics_br = evaluate_predictions(y_br[mask_test_br], y_prev_br)

print(f"\nValutazione SVM su stazioni di test BR:")
print(f"  RMSE: {metrics_br['rmse']:.3f}")
print(f"  MAE:  {metrics_br['mae']:.3f}")
print(f"  R2:   {metrics_br['r2']:.3f}")
print(f"  (calcolate su {metrics_br['n_valutati']} osservazioni valide)")

mask_train_fg, mask_test_fg = train_test_split_by_station(id_fg, test_size=0.2, seed=13)

model_fg = RFSTK(model_type='xgboost',model_params=dict(n_estimators=20, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_fg.fit(X_fg[mask_train_fg], y_fg[mask_train_fg], id_fg[mask_train_fg], ind_t_fg[mask_train_fg],coord_fg["stazione"].to_numpy(), coord_fg["latitudine"].to_numpy(),coord_fg["longitudine"].to_numpy(),)
print(model_fg.summary()) 
y_prev_fg = model_fg.predict(X_fg[mask_test_fg], id_fg[mask_test_fg], ind_t_fg[mask_test_fg])


metrics_fg = evaluate_predictions(y_fg[mask_test_fg], y_prev_fg)

print(f"\nValutazione SVM su stazioni di test FG:")
print(f"  RMSE: {metrics_fg['rmse']:.3f}")
print(f"  MAE:  {metrics_fg['mae']:.3f}")
print(f"  R2:   {metrics_fg['r2']:.3f}")
print(f"  (calcolate su {metrics_fg['n_valutati']} osservazioni valide)")

mask_train_le, mask_test_le = train_test_split_by_station(id_le, test_size=0.2, seed=13)

model_le = RFSTK(model_type='xgboost',model_params=dict(n_estimators=25, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_le.fit(X_le[mask_train_le], y_le[mask_train_le], id_le[mask_train_le], ind_t_le[mask_train_le],coord_le["stazione"].to_numpy(), coord_le["latitudine"].to_numpy(),coord_le["longitudine"].to_numpy(),)
print(model_le.summary()) 
y_prev_le = model_le.predict(X_le[mask_test_le], id_le[mask_test_le], ind_t_le[mask_test_le])

metrics_le = evaluate_predictions(y_le[mask_test_le], y_prev_le)

print(f"\nValutazione SVM su stazioni di test LE:")
print(f"  RMSE: {metrics_le['rmse']:.3f}")
print(f"  MAE:  {metrics_le['mae']:.3f}")
print(f"  R2:   {metrics_le['r2']:.3f}")
print(f"  (calcolate su {metrics_le['n_valutati']} osservazioni valide)")

mask_train_ta, mask_test_ta = train_test_split_by_station(id_ta, test_size=0.2, seed=13)

model_ta = RFSTK(model_type='xgboost',model_params=dict(n_estimators=50, max_depth=1),kriging_params=dict(max_neighbors=30*10, time_window=7.0),fixed_lambda=1., use_km_projection=True )
model_ta.fit(X_ta[mask_train_ta], y_ta[mask_train_ta], id_ta[mask_train_ta], ind_t_ta[mask_train_ta],coord_ta["stazione"].to_numpy(), coord_ta["latitudine"].to_numpy(),coord_ta["longitudine"].to_numpy(),)
print(model_ta.summary()) 
y_prev_ta = model_ta.predict(X_ta[mask_test_ta], id_ta[mask_test_ta], ind_t_ta[mask_test_ta])

metrics_ta = evaluate_predictions(y_ta[mask_test_ta], y_prev_ta)

print(f"\nValutazione SVM su stazioni di test TA:")
print(f"  RMSE: {metrics_ta['rmse']:.3f}")
print(f"  MAE:  {metrics_ta['mae']:.3f}")
print(f"  R2:   {metrics_ta['r2']:.3f}")
print(f"  (calcolate su {metrics_ta['n_valutati']} osservazioni valide)")

63
{'model_type': 'xgboost', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 416.2510457285646, 'nugget': 41.62510457285646, 'partial_sill': 374.6259411557081, 'theta_s_km': 3.9992166758035568, 'theta_t_days': 27.569141992817517, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test BA:
  RMSE: 14.245
  MAE:  11.503
  R2:   0.114
  (calcolate su 5046 osservazioni valide)
{'model_type': 'xgboost', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 277.3302033490683, 'nugget': 27.733020334906833, 'partial_sill': 249.5971830141615, 'theta_s_km': 6.853149683933669, 'theta_t_days': 15.666813969345538, 'lambda_shrinkage': 1.0, 'cv_r2_by_lambda': None}

Valutazione SVM su stazioni di test BT:
  RMSE: 11.814
  MAE:  7.850
  R2:   0.439
  (calcolate su 714 osservazioni valide)
{'model_type': 'xgboost', 'oob_score_rf': None, "unita' spaziali": 'km', 'total_var': 154.73166481027278, 'nugget': 15.473166481027278, 'partial_sill': 139.25849832924